# Chapter 2: Data Structures & Algorithms
## Notebook 02 - Intermediate

Stacks, queues, hash tables, and sorting - the workhorses of efficient code.

**What you'll learn:**
- Stacks and queues (LIFO/FIFO)
- Hash tables and collision handling
- Sorting algorithms: merge sort, quicksort
- Choosing the right data structure

**Time estimate:** 2 hours

---
*Generated by Berta AI | Created by Luigi Giacobbe*

## 1. Stacks (LIFO)

Last-In-First-Out. Used for: undo operations, backtracking, parsing expressions, DFS, call stacks in recursion.

In [ ]:
class Stack:
    """A stack implementation using a Python list."""
    
    def __init__(self):
        self._items = []
    
    def push(self, item):
        self._items.append(item)
    
    def pop(self):
        if self.is_empty():
            raise IndexError("Pop from empty stack")
        return self._items.pop()
    
    def peek(self):
        if self.is_empty():
            raise IndexError("Peek at empty stack")
        return self._items[-1]
    
    def is_empty(self):
        return len(self._items) == 0
    
    def __len__(self):
        return len(self._items)
    
    def __repr__(self):
        return f"Stack({self._items})"


# Practical: balanced parentheses checker
# (used in expression parsing, code validation)
def is_balanced(expression):
    """Check if brackets in an expression are balanced."""
    stack = Stack()
    pairs = {')': '(', ']': '[', '}': '{'}
    
    for char in expression:
        if char in '([{':
            stack.push(char)
        elif char in ')]}':
            if stack.is_empty() or stack.pop() != pairs[char]:
                return False
    
    return stack.is_empty()


tests = [
    "model(layers[0].forward({input: x}))",
    "loss = (pred - target) ** 2",
    "broken([)",
    "{config: {lr: 0.01, epochs: [10, 20]}}",
]

for expr in tests:
    result = 'balanced' if is_balanced(expr) else 'UNBALANCED'
    print(f"  {result:>12}: {expr}")

## 2. Queues (FIFO)

First-In-First-Out. Used for: BFS, task scheduling, data streaming, batch processing, message queues.

In [ ]:
from collections import deque


class TaskQueue:
    """A priority-aware task queue for ML job scheduling."""
    
    def __init__(self):
        self.high = deque()
        self.normal = deque()
        self.low = deque()
    
    def enqueue(self, task, priority="normal"):
        q = {"high": self.high, "normal": self.normal, "low": self.low}[priority]
        q.append(task)
    
    def dequeue(self):
        for q in [self.high, self.normal, self.low]:
            if q:
                return q.popleft()
        return None
    
    @property
    def size(self):
        return len(self.high) + len(self.normal) + len(self.low)


# Simulate an ML job scheduler
scheduler = TaskQueue()
scheduler.enqueue("train_model_v2", "normal")
scheduler.enqueue("fix_data_pipeline", "high")
scheduler.enqueue("generate_report", "low")
scheduler.enqueue("retrain_urgent", "high")
scheduler.enqueue("evaluate_model", "normal")

print("Processing ML jobs:")
while scheduler.size > 0:
    job = scheduler.dequeue()
    print(f"  Running: {job}")

## 3. Sorting Algorithms

Sorting is fundamental to data processing. Understanding *how* sorting works helps you choose the right approach for your data.

In [ ]:
def merge_sort(arr):
    """O(n log n) sorting - divide and conquer. Stable sort."""
    if len(arr) <= 1:
        return arr
    
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    
    return merge(left, right)


def merge(left, right):
    """Merge two sorted lists into one sorted list."""
    result = []
    i = j = 0
    
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    
    result.extend(left[i:])
    result.extend(right[j:])
    return result


def quicksort(arr):
    """O(n log n) average sorting - partition-based. In-place possible."""
    if len(arr) <= 1:
        return arr
    
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    
    return quicksort(left) + middle + quicksort(right)


# Demonstrate
import random
random.seed(42)
data = [random.randint(0, 100) for _ in range(15)]

print(f"Original:   {data}")
print(f"Merge sort: {merge_sort(data)}")
print(f"Quicksort:  {quicksort(data)}")
print(f"Built-in:   {sorted(data)}")

# Performance comparison
import time
large_data = [random.randint(0, 1_000_000) for _ in range(10_000)]

for name, func in [("merge_sort", merge_sort), ("quicksort", quicksort), ("sorted()", sorted)]:
    start = time.perf_counter()
    func(large_data)
    elapsed = time.perf_counter() - start
    print(f"  {name:>12}: {elapsed*1000:.2f} ms on {len(large_data):,} items")

## 4. Choosing the Right Data Structure

| Problem | Best Structure | Why |
|---------|---------------|-----|
| Store training batches in order | List | Index access, append |
| Fast lookup of labels | Dict/Set | O(1) lookup |
| Processing jobs in order | Queue (deque) | FIFO |
| Undo/redo history | Stack | LIFO |
| Unique vocabulary | Set | Auto-dedup |
| Model config | Dict | Key-value pairs |
| Fixed-shape data | Tuple | Immutable, hashable |
| Priority scheduling | Heap/PriorityQueue | Efficient min/max |

In [ ]:
# Real-world pattern: choosing structures for an ML pipeline

from collections import Counter, defaultdict, OrderedDict
import heapq

# Counter for word frequencies (NLP)
text = "the model predicts the label and the model learns from the data"
vocab = Counter(text.split())
print(f"Vocabulary: {dict(vocab.most_common(5))}")

# defaultdict for grouping (clustering results)
predictions = [("cat", 0.9), ("dog", 0.8), ("cat", 0.7), ("bird", 0.6), ("dog", 0.95)]
by_class = defaultdict(list)
for label, confidence in predictions:
    by_class[label].append(confidence)

print(f"\nGrouped predictions:")
for label, scores in by_class.items():
    print(f"  {label}: {scores} (avg: {sum(scores)/len(scores):.2f})")

# Heap for top-K selection (common in recommendation systems)
all_scores = [(0.95, "model_A"), (0.87, "model_B"), (0.92, "model_C"),
              (0.89, "model_D"), (0.91, "model_E"), (0.88, "model_F")]

top_3 = heapq.nlargest(3, all_scores)
print(f"\nTop 3 models: {[(name, f'{score:.0%}') for score, name in top_3]}")

## What's Next?

In **Notebook 03 (Advanced)**, we'll cover:
- Trees and tree traversals
- Graphs and graph algorithms (BFS, DFS)
- Dynamic programming basics
- Capstone: building a simple search index

---
*Generated by Berta AI | Created by Luigi Giacobbe*